# ADNI Multimodal Alzheimer’s Disease EDA

This notebook consolidates the original exploratory notebooks into one clean, narrative-driven analysis.

The workflow covers:

1. Raw ADNI tabular data inspection  
2. MRI/DICOM inventory and quality checks  
3. Patient overlap across `ADNIAlpha` and `ADNIBeta` image folders  
4. Diagnosis, cognition, plasma biomarker, MRI, and APOE genetics merge  
5. Longitudinal cleanup over a 0–36 month window  
6. Feature selection and imputation  
7. Descriptive EDA by diagnosis group  
8. Multimodal coverage and longitudinal MRI trends  

Diagnosis classes:

- `1 = CN` — Cognitively Normal  
- `2 = MCI` — Mild Cognitive Impairment  
- `3 = AD` — Alzheimer’s Disease

## 0. Setup

This section centralizes imports, paths, plotting defaults, and dataset filenames.  
The final notebook assumes the same relative project structure used in the original notebooks.

In [ ]:
import os
import re
import warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import seaborn as sns
except ImportError:
    sns = None

try:
    import pydicom
except ImportError:
    pydicom = None

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 20)

PROJECT_ROOT = Path(os.getcwd()).parent
DATA_DIR = PROJECT_ROOT / "data"
BASE_DIR = DATA_DIR / "01_raw" / "ADNI"
INTERMEDIATE_DIR = DATA_DIR / "02_intermediate"
PRIMARY_DIR = DATA_DIR / "03_primary"
REPORTING_DIR = DATA_DIR / "08_reporting"

INTERMEDIATE_DIR.mkdir(parents=True, exist_ok=True)
REPORTING_DIR.mkdir(parents=True, exist_ok=True)

FILES = {
    "diagnosis": "DXSUM.csv",
    "brain_vol": "ADSP_BIOMARKER_SCORE.csv",
    "cognition": "ADSP_COGN_SCORE.csv",
    "genetics": "APOERES.csv",
    "plasma": "PLASMA.csv",
}

ALPHA_DIR = BASE_DIR / "ADNIAlpha"
BETA_DIR = BASE_DIR / "ADNIBeta"

MERGED_IMPUTED_CSV = INTERMEDIATE_DIR / "adni_merged_imputed.csv"
FINAL_MODELING_CSV = INTERMEDIATE_DIR / "adni_final_for_modeling.csv"
CLEANED_FEATURE_CSV = INTERMEDIATE_DIR / "adni_cleaned_feature_selected.csv"
LINKED_CSV = PRIMARY_DIR / "adni_multimodal_linked.csv"

DX_LABELS = {1.0: "CN", 2.0: "MCI", 3.0: "AD"}
DX_LABELS_LONG = {1.0: "CN (Normal)", 2.0: "MCI", 3.0: "AD (Alzheimer's)"}
DX_ORDER = [1.0, 2.0, 3.0]

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "font.size": 11,
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
})

## 1. Raw Tabular Dataset Inspection

The first step is to inspect the major ADNI tabular files before merging.  
For each file, we report:

- shape  
- duplicate rows  
- missing-value burden  
- first few rows  

This provides a quick data quality baseline before downstream cleaning.

In [ ]:
def load_and_inspect_csv(file_path: Path, name: str) -> pd.DataFrame:

    print(f"File: {name.upper()}")
    
    try:
        df = pd.read_csv(file_path)
        
        # Shape analysis
        rows, cols = df.shape
        print(f"Shape: {rows} rows, {cols} columns")
        
        # Duplicate check
        duplicates = df.duplicated().sum()
        print(f"Duplicate Rows: {duplicates}")
        
        # Missing value analysis
        missing_counts = df.isnull().sum()
        
        # Filter for columns that actually have missing values
        missing_data = missing_counts[missing_counts > 0]
        
        if not missing_data.empty:
            # Calculate percentage
            missing_percentage = (missing_data / rows) * 100
            
            # Create a summary dataframe
            missing_summary = pd.DataFrame({
                'Missing Count': missing_data,
                'Percentage': missing_percentage
            })
            
            # Sort by percentage descending to see worst offenders first
            missing_summary = missing_summary.sort_values(by='Percentage', ascending=False)
            
            print(f"Columns with Missing Values ({len(missing_summary)} columns):")
            display(missing_summary.head(15).style.format({'Percentage': '{:.2f}%'}))
        else:
            print("No missing values found.")

        # Quick look
        print("First 3 rows:")
        display(df.head(3))
        
        return df

    except FileNotFoundError:
        print(f"Can't find file at {file_path}")
        return None

In [ ]:
def inspect_dicom_structure(directory: Path):
    """
    Walks through a directory to count DICOM files and visualize structure.
    """
    print(f"File structure of: {directory.name}")
    
    if not directory.exists():
        print(f"Directory not found: {directory}")
        return

    dcm_count = 0
    
    # Walk through directory tree
    for root, dirs, files in os.walk(directory):
        level = root.replace(str(directory), '').count(os.sep)
        indent = ' ' * 4 * (level)
        folder_name = os.path.basename(root)
        
        # Count .dcm files in this folder
        current_dcms = [f for f in files if f.endswith('.dcm')]
        count = len(current_dcms)
        dcm_count += count
        
        # Only print folders if they are relevant
        if level < 3: 
            print(f"{indent}|-- {folder_name}/ ({count} .dcm files)")

    print(f"\nTotal DICOM files found: {dcm_count}")

In [ ]:
# Dictionary to store loaded dataframes for easy access later
dfs = {}

# Inspect CSV data
for friendly_name, filename in FILES.items():
    full_path = BASE_DIR / filename
    dfs[friendly_name] = load_and_inspect_csv(full_path, friendly_name)

# Inspect image data structure
inspect_dicom_structure(PATIENT_IMAGE_DIR)

In [ ]:
# Load and inspect the core tabular datasets.
# Comment this cell out if you only want to run downstream steps from saved intermediate files.

datasets = {}
for name, filename in FILES.items():
    datasets[name] = load_and_inspect_csv(BASE_DIR / filename, name)

### Initial Inspection Takeaways

The raw files differ substantially in structure and completeness:

- Diagnosis data contains many longitudinal rows and several diagnosis-specific fields.
- MRI-derived brain volume data is high-dimensional.
- APOE genetics is static at the patient level.
- Plasma biomarker availability is incomplete and must be handled carefully.
- Cognition and diagnosis are longitudinal and should be merged by `RID` and `VISCODE`.

## 2. MRI/DICOM Image Inventory

The raw image folders are inspected separately from the tabular MRI-derived features.  
The goal is to understand scan availability and patient overlap across `ADNIAlpha` and `ADNIBeta`.

In [ ]:
def parse_adni_directory(base_path, folder_name):
    """
    Traverses the ADNI directory to find DICOM files and extracts 
    Patient ID and scan date from the folder structure.
    """
    print(f"Currently in {folder_name}")
    data = []
    
    if not base_path.exists():
        print(f"Can't find directory: {base_path}")
        return pd.DataFrame()

    # Regex to identify patients
    ptid_pattern = re.compile(r'\d{3}_S_\d{4}')
    
    # Regex to identify dates
    date_pattern = re.compile(r'\d{4}-\d{2}-\d{2}')

    for root, dirs, files in os.walk(base_path):
        # We only care about leaf folders that contain .dcm files
        dcm_files = [f for f in files if f.endswith('.dcm')]
        
        if dcm_files:
            path_parts = Path(root).parts
            
            # Extract PTID
            ptid = "Unknown"
            for part in path_parts:
                if ptid_pattern.match(part):
                    ptid = part
                    break
            
            # Extract Date
            scan_date = "Unknown"
            for part in path_parts:
                match = date_pattern.search(part) 
                if match:
                    scan_date = match.group(0)
                    break
            
            data.append({
                'Folder': folder_name,
                'PTID': ptid,
                'Scan_Date': scan_date,
                'Image_Count': len(dcm_files),
                'Full_Path': root
            })

    df = pd.DataFrame(data)
    print(f"Found {len(df)} scan directories in {folder_name}")
    return df

In [ ]:
df_alpha = parse_adni_directory(ALPHA_DIR, "ADNIAlpha")
df_beta = parse_adni_directory(BETA_DIR, "ADNIBeta")

image_inventory = pd.concat([df_alpha, df_beta], ignore_index=True)

if not image_inventory.empty:
    alpha_patients = set(df_alpha["PTID"].dropna().unique())
    beta_patients = set(df_beta["PTID"].dropna().unique())

    image_summary = {
        "Total scan directories": len(image_inventory),
        "Total unique MRI patients": image_inventory["PTID"].nunique(),
        "Patients in ADNIAlpha": len(alpha_patients),
        "Patients in ADNIBeta": len(beta_patients),
        "Overlapping patients": len(alpha_patients & beta_patients),
        "Alpha-only patients": len(alpha_patients - beta_patients),
        "Beta-only patients": len(beta_patients - alpha_patients),
    }

    display(pd.DataFrame.from_dict(image_summary, orient="index", columns=["Count"]))
    display(image_inventory[["Folder", "PTID", "Scan_Date", "Image_Count"]].head())
else:
    print("No image data found.")

### Optional: Patient Lists for ADNIAlpha / ADNIBeta Overlap

The full patient lists are usually too long for the final EDA narrative, so this section keeps them as exportable QA tables instead of printing large lists.

In [ ]:
def get_unique_ptids(base_path: Path) -> set:
    """Scan a directory and return unique ADNI-format patient IDs."""
    ptids = set()
    if not base_path.exists():
        print(f"Can't find {base_path}.")
        return ptids

    ptid_pattern = re.compile(r"\d{3}_S_\d{4}")

    for root, dirs, files in os.walk(base_path):
        match = ptid_pattern.search(str(root))
        if match:
            ptids.add(match.group(0))

    return ptids

alpha_ptids = get_unique_ptids(ALPHA_DIR)
beta_ptids = get_unique_ptids(BETA_DIR)

overlapping_patients = sorted(alpha_ptids & beta_ptids)
alpha_only_patients = sorted(alpha_ptids - beta_ptids)
beta_only_patients = sorted(beta_ptids - alpha_ptids)

max_len = max(len(overlapping_patients), len(alpha_only_patients), len(beta_only_patients), 1)

def pad_list(lst, length):
    return lst + [None] * (length - len(lst))

patient_overlap_df = pd.DataFrame({
    "Overlapping_PTIDs": pad_list(overlapping_patients, max_len),
    "Alpha_Only_PTIDs": pad_list(alpha_only_patients, max_len),
    "Beta_Only_PTIDs": pad_list(beta_only_patients, max_len),
})

display(pd.DataFrame({
    "Group": ["Overlapping", "Alpha only", "Beta only"],
    "Patients": [len(overlapping_patients), len(alpha_only_patients), len(beta_only_patients)]
}))

patient_overlap_df.to_csv(INTERMEDIATE_DIR / "adni_alpha_beta_patient_overlap.csv", index=False)

## 3. Optional MRI Scan QA Example

This optional step confirms that raw DICOM files are readable and can be ordered into a coherent MRI volume.  
It uses one representative patient as a quality check and does **not** directly feed the tabular modeling dataset.

In [ ]:
def load_scan(path):
    slices = []
    print(f"Loading scan from: {path}")
    for s in os.listdir(path):
        if s.endswith(".dcm"):
            try:
                ds = pydicom.dcmread(os.path.join(path, s))
                slices.append(ds)
            except Exception:
                pass
    if not slices:
        return []
    
    # Sort by instance number to get 3D order
    try:
        slices.sort(key=lambda x: int(x.InstanceNumber))
    except AttributeError:
        slices.sort(key=lambda x: float(x.ImagePositionPatient[2]))
        
    return slices

In [ ]:
def find_deepest_dicom_folder(base_path, ptid):
    target_dir = None
    max_files = 0
    for root, dirs, files in os.walk(base_path):
        if ptid in root:
            dcm_count = len([f for f in files if f.endswith(".dcm")])
            if dcm_count > max_files:
                max_files = dcm_count
                target_dir = root
    return target_dir

In [ ]:
def generate_upright_mri():
    scan_dir = find_deepest_dicom_folder(BASE_SEARCH_DIR, TARGET_PTID)
    if not scan_dir:
        print("ERROR FOR SCAN!")
        return

    slices = load_scan(scan_dir)
    
    # Select middle slice
    mid_idx = len(slices) // 2
    mid_slice = slices[mid_idx].pixel_array
    
    # Rotate 90 degrees counter clockwise but by 4x I think
    corrected_slice = np.rot90(mid_slice, k=4)


    # Contrast enhancement
    vmin = np.percentile(corrected_slice, 1)
    vmax = np.percentile(corrected_slice, 99)

    # Visualize
    plt.figure(figsize=(8, 8))
    plt.imshow(corrected_slice, cmap='gray', vmin=vmin, vmax=vmax)
    
    plt.axis('off')
    plt.title(f"Sagittal MRI of Patient: {TARGET_PTID}", fontsize=14, weight='bold')
    
    save_path = OUTPUT_DIR / "mri_sample_visual.png"
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()

if __name__ == "__main__":
    generate_upright_mri()

In [ ]:
# Example usage:
TARGET_PTID = "941_S_7051"
BASE_SEARCH_DIR = ALPHA_DIR

if pydicom is None:
    print("pydicom is not installed, so DICOM QA visualization is skipped.")
else:
    target_dir = find_deepest_dicom_folder(BASE_SEARCH_DIR, TARGET_PTID)
    if target_dir:
        slices = load_scan(target_dir)
        print(f"Loaded {len(slices)} DICOM slices for {TARGET_PTID}.")
    else:
        print(f"No DICOM folder found for {TARGET_PTID}.")

## 4. Cohort Construction, Merge, and Initial Imputation

This section creates the first merged longitudinal dataset.

Merge strategy:

- Diagnosis, cognition, plasma, and MRI-derived volumes merge on `RID` and `VISCODE`.
- APOE genotype is treated as a static patient-level feature and merges on `RID`.
- APOE missingness is filled with the genotype mode.
- Plasma and cognition features are imputed by diagnosis-group median, with a global median fallback.

MRI feature imputation is intentionally deferred until after selecting key MRI variables.

In [ ]:
COLS_TO_KEEP = {
    "diagnosis": ["RID", "PTID", "VISCODE", "EXAMDATE", "DIAGNOSIS", "DXAD", "DXMCI", "DXNORM"],
    "genetics": ["RID", "GENOTYPE"], 
    "plasma": ["RID", "VISCODE", "pT217_F", "AB42_F", "AB40_F", "NfL_Q", "GFAP_Q"],
    "cognition": ["RID", "VISCODE", "PHC_MEM", "PHC_EXF", "PHC_LAN"], 
}

In [ ]:
def load_data(file_path: Path) -> pd.DataFrame:
    if not file_path.exists():
        print(f"File not found at {file_path}")
        return pd.DataFrame()
    return pd.read_csv(file_path, low_memory=False)

In [ ]:
def preprocess_genetics(df_gen: pd.DataFrame) -> pd.DataFrame:
    """
    Genetics is a static trait. We extract one genotype per RID.
    """
    if df_gen.empty:
        print("Genetics dataframe is empty.")
        return pd.DataFrame(columns=['RID', 'GENOTYPE'])

    # Drop rows with no Genotype
    df_gen = df_gen.dropna(subset=['GENOTYPE'])
    
    if 'APTESTDT' in df_gen.columns:
        df_gen = df_gen.sort_values('APTESTDT')
    
    # Drop duplicates to keep one row per Patient (RID)
    df_static = df_gen[['RID', 'GENOTYPE']].drop_duplicates(subset=['RID'], keep='first')
    
    print(f"Unique Genotypes extracted for {len(df_static)} patients.")
    return df_static

In [ ]:
def clean_diagnosis(df_dx: pd.DataFrame) -> pd.DataFrame:
    """
    Cleans the target variable.
    """
    if df_dx.empty:
        print("Diagnosis dataframe is empty.")
        return pd.DataFrame()

    # Keep only relevant columns
    df_dx = df_dx[COLS_TO_KEEP['diagnosis']].copy()
    
    # Drop rows where global diagnosis is missing
    initial_len = len(df_dx)
    df_dx = df_dx.dropna(subset=['DIAGNOSIS'])
    print(f"Dropped {initial_len - len(df_dx)} rows with missing Diagnosis.")
    
    # Standardize Dates
    df_dx['EXAMDATE'] = pd.to_datetime(df_dx['EXAMDATE'])
    
    return df_dx

In [ ]:
def impute_and_merge():
    """
    Main execution pipeline: Load -> Clean -> Merge -> Impute -> Save to 02_intermediate.
    """
    # 1. Load Data
    dfs = {name: load_data(BASE_DIR / fname) for name, fname in FILES.items()}
    
    # Stop if critical files missing
    if dfs['diagnosis'].empty:
        return

    # 2. Process static data
    df_genetics_clean = preprocess_genetics(dfs['genetics'])
    
    # 3. Process longitudinal data
    df_main = clean_diagnosis(dfs['diagnosis'])
    
    # 4. Merge longitudinal modalities left join
    
    # Merge Cognition
    if not dfs['cognition'].empty:
        df_cog = dfs['cognition'][COLS_TO_KEEP['cognition']]
        df_main = df_main.merge(df_cog, on=['RID', 'VISCODE'], how='left')
    
    # Merge plasma 
    if not dfs['plasma'].empty:
        df_plasma = dfs['plasma'][COLS_TO_KEEP['plasma']]
        df_main = df_main.merge(df_plasma, on=['RID', 'VISCODE'], how='left')
    
    #  Merge brain volume
    if not dfs['brain_vol'].empty:
        # Dropping metadata columns to avoid collision, keeping useful features
        df_vol = dfs['brain_vol'].drop(columns=['update_stamp', 'PTID', 'VISCODE2'], errors='ignore')
        df_main = df_main.merge(df_vol, on=['RID', 'VISCODE'], how='left')
    
    # 5. Merge static data
    df_main = df_main.merge(df_genetics_clean, on='RID', how='left')
    
    print(f"\nMerged Dataset Shape: {df_main.shape}")
    
    # 6. Imputation Strategy
    # A. Fill genotype with mode
    if 'GENOTYPE' in df_main.columns:
        geno_mode = df_main['GENOTYPE'].mode()[0]
        df_main['GENOTYPE'] = df_main['GENOTYPE'].fillna(geno_mode)
        print(f"Imputed missing Genotypes with mode: {geno_mode}")
    
    # B. Fill Continuous Modalities plasma and cognition with median
    continuous_cols = [
        'pT217_F', 'AB42_F', 'AB40_F', 'NfL_Q', 'GFAP_Q', # Plasma
        'PHC_MEM', 'PHC_EXF', 'PHC_LAN' # Cognition
    ]
    
    for col in continuous_cols:
        if col in df_main.columns:
            # Calculate missingness before
            missing_pct = df_main[col].isnull().mean()
            if missing_pct > 0:
                # Impute by grouped median
                # If diagnosis is missing it ignores
                df_main[col] = df_main[col].fillna(df_main.groupby('DIAGNOSIS')[col].transform('median'))
                
                # Fallback: If a diagnosis group is entirely NaN for that col
                df_main[col] = df_main[col].fillna(df_main[col].median())
    output_path = INTERMEDIATE_DIR / "adni_merged_imputed.csv"
    df_main.to_csv(output_path, index=False)
    print(f"Merged file saved to: {output_path}")
    print("First 5 rows of merged data:")
    display(df_main.head())

In [ ]:
# Run merge + initial imputation.
# This writes ../data/02_intermediate/adni_merged_imputed.csv

impute_and_merge()

## 5. Longitudinal Cleanup and Feature Selection

The merged dataset is restricted to the first 36 months from each patient’s baseline exam date.  
A compact feature set is then selected for modeling and EDA.

MRI imputation occurs here after key MRI regions are selected, reducing noise from the full high-dimensional MRI table.

In [ ]:
# 1. Define key MRI features
KEY_MRI_FEATURES = [
    "Left.Hippocampus_combat", "Right.Hippocampus_combat",
    "lh_entorhinal_volume_combat", "rh_entorhinal_volume_combat",
    "lh_entorhinal_thickness_combat", "rh_entorhinal_thickness_combat",
    "Left.Lateral.Ventricle_combat", "Right.Lateral.Ventricle_combat",
    "BrainSegVol_combat", "EstimatedTotalIntraCranialVol_combat"
]

In [ ]:
# 2. Define core features
CORE_FEATURES = [
    "RID", "PTID", "VISCODE", "EXAMDATE", 
    "DIAGNOSIS", # 1=CN, 2=MCI, 3=AD
    "GENOTYPE", 
    "pT217_F", "AB42_F", "AB40_F", "NfL_Q", "GFAP_Q",
    "PHC_MEM", "PHC_EXF", "PHC_LAN"
]

In [ ]:
def calculate_months_from_baseline(df: pd.DataFrame) -> pd.DataFrame:
    """
    Calculates precise months from baseline using actual dates, 
    then filters for the 36 month cap
    """
    # Create a deep copy to prevent SettingWithCopyWarning
    df = df.copy()
    
    # Solid date format
    df['EXAMDATE'] = pd.to_datetime(df['EXAMDATE'], errors='coerce')
    df = df.dropna(subset=['EXAMDATE'])

    # Find baseline date for each patient
    # We take the minimum date for each patient as their 'Day 0'
    baseline_dates = df.groupby('RID')['EXAMDATE'].transform('min')
    
    # Calculate difference in days
    df['Days_From_Baseline'] = (df['EXAMDATE'] - baseline_dates).dt.days
    
    # Convert to months and round to nearest integer
    df['Month'] = (df['Days_From_Baseline'] / 30.44).round().astype(int)
    
    # 36 MONTH CAP
    initial_count = len(df)
    df = df[df['Month'] <= 36]
    dropped_count = initial_count - len(df)
    
    print(f"36 Month Cap: Dropped {dropped_count} rows beyond 3 years.")
    print(f"Remaining Data Points: {len(df)}")
    
    return df

In [ ]:
def feature_selection_and_imputation(df: pd.DataFrame) -> pd.DataFrame:
    """
    Selects columns and imputes missing MRI data using DG medians
    """    
    # 1. Select Columns
    available_mri = [c for c in KEY_MRI_FEATURES if c in df.columns]
    
    # 2: Explicitly include month in the features to keep
    cols_to_keep = CORE_FEATURES + available_mri + ['Month']
    
    df_clean = df[cols_to_keep].copy()
    
    print(f"Reduced feature set to {df_clean.shape[1]} columns.")
    
    # 2. Impute missing MRI features
    for col in available_mri:
        missing_count = df_clean[col].isnull().sum()
        if missing_count > 0:
            # Impute by Diagnosis group which will help retain signal
            df_clean[col] = df_clean[col].fillna(
                df_clean.groupby('DIAGNOSIS')[col].transform('median')
            )
            # Fallback is the global median
            df_clean[col] = df_clean[col].fillna(df_clean[col].median())
            
            print(f"Imputed {missing_count} missing values in {col}")
            
    return df_clean

In [ ]:
def main():
    if not INPUT_FILE.exists():
        print(f"Can't find file at {INPUT_FILE}")
        return
        
    df = pd.read_csv(INPUT_FILE, low_memory=False)
    print(f"Loaded Data: {df.shape}")
    
    # 1. Calculate time & cap at 36 months
    df = calculate_months_from_baseline(df)
    
    # 2. Clean & impute
    df_final = feature_selection_and_imputation(df)
    
    # 3. Final Inspection
    print(f"Shape: {df_final.shape}")
    print(f"Unique Months: {sorted(df_final['Month'].unique())}")
    
    # 3: Add patient count
    unique_patients = df_final['RID'].nunique()
    print(f"Total unique patients based on RID: {unique_patients}")
    
    # Check for NaNs
    missing_total = df_final.isnull().sum().sum()
    print(f"Total missing values: {missing_total}")
    
    # 4. Save
    df_final.to_csv(OUTPUT_FILE, index=False)
    print(f"Saved cleaned dataset to: {OUTPUT_FILE}")

In [ ]:
# Run longitudinal cleanup + feature selection.
# The cleaned final dataset is saved as adni_final_for_modeling.csv.

main()

## 6. Descriptive EDA by Diagnosis

This section summarizes biomarkers, cognition, APOE genotype risk, and selected MRI features across diagnosis groups.

In [ ]:
# Load final modeling dataset.
# If adni_final_for_modeling.csv does not exist yet, fall back to adni_cleaned_feature_selected.csv.

if FINAL_MODELING_CSV.exists():
    df = pd.read_csv(FINAL_MODELING_CSV)
elif CLEANED_FEATURE_CSV.exists():
    df = pd.read_csv(CLEANED_FEATURE_CSV)
else:
    raise FileNotFoundError("No final modeling file found. Run Sections 4 and 5 first.")

diagnosis_map = {1.0: "CN", 2.0: "MCI", 3.0: "AD"}
df["Diagnosis_Label"] = df["DIAGNOSIS"].map(diagnosis_map)
df["Diagnosis_Label"] = pd.Categorical(df["Diagnosis_Label"], categories=["CN", "MCI", "AD"], ordered=True)

def categorize_genotype(g):
    if pd.isna(g):
        return "Unknown"
    g = str(g)
    if "4" in g:
        return "Risk Homozygous" if g == "4/4" else "Risk Heterozygous"
    return "Neutral"

df["Genotype_Risk"] = df["GENOTYPE"].apply(categorize_genotype)

biomarkers = [
    "pT217_F", "AB42_F", "AB40_F", "NfL_Q", "GFAP_Q",
    "PHC_MEM", "PHC_EXF", "PHC_LAN",
    "Left.Hippocampus_combat", "Right.Hippocampus_combat",
    "lh_entorhinal_volume_combat", "rh_entorhinal_volume_combat",
    "lh_entorhinal_thickness_combat", "rh_entorhinal_thickness_combat",
    "Left.Lateral.Ventricle_combat", "Right.Lateral.Ventricle_combat",
    "BrainSegVol_combat", "EstimatedTotalIntraCranialVol_combat",
]

biomarkers = [c for c in biomarkers if c in df.columns]

print(f"Final dataset shape: {df.shape}")
print(f"Unique patients: {df['PTID'].nunique():,}")
print(f"Month range: {df['Month'].min()}–{df['Month'].max()}")
display(df["Diagnosis_Label"].value_counts().sort_index().to_frame("Observations"))

In [ ]:
# Summary table: mean ± standard deviation by diagnosis.

summary_df = df.groupby("Diagnosis_Label", observed=False)[biomarkers].agg(["mean", "std"]).transpose()

table_data = []
for feature in biomarkers:
    row = [feature]
    for diag in ["CN", "MCI", "AD"]:
        mean_val = summary_df.loc[(feature, "mean"), diag]
        std_val = summary_df.loc[(feature, "std"), diag]

        if abs(mean_val) > 100:
            val_str = f"{mean_val:,.0f} ± {std_val:,.0f}"
        elif abs(mean_val) > 1:
            val_str = f"{mean_val:.2f} ± {std_val:.2f}"
        else:
            val_str = f"{mean_val:.3f} ± {std_val:.3f}"
        row.append(val_str)
    table_data.append(row)

description_table = pd.DataFrame(table_data, columns=["Feature", "CN", "MCI", "AD"])
display(description_table)

In [ ]:
# Diagnosis distribution

ax = df["Diagnosis_Label"].value_counts().sort_index().plot(kind="bar", figsize=(7, 4))
ax.set_title("Diagnosis Distribution")
ax.set_xlabel("Diagnosis")
ax.set_ylabel("Observations")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# APOE genotype risk distribution

risk_counts = pd.crosstab(df["Diagnosis_Label"], df["Genotype_Risk"], normalize="index") * 100
display(risk_counts.round(2))

risk_counts.plot(kind="bar", figsize=(9, 5))
plt.title("APOE Genotype Risk Distribution by Diagnosis")
plt.xlabel("Diagnosis")
plt.ylabel("Percent within diagnosis")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Feature distributions for representative biomarkers/cognition/MRI.

features_to_plot = [
    "pT217_F",
    "AB42_F",
    "AB40_F",
    "PHC_MEM",
    "Left.Hippocampus_combat",
    "Right.Lateral.Ventricle_combat",
]

features_to_plot = [c for c in features_to_plot if c in df.columns]

for feature in features_to_plot:
    plt.figure(figsize=(7, 4))
    if sns is not None:
        sns.boxplot(data=df, x="Diagnosis_Label", y=feature)
    else:
        df.boxplot(column=feature, by="Diagnosis_Label")
        plt.suptitle("")
    plt.title(f"{feature} by Diagnosis")
    plt.xlabel("Diagnosis")
    plt.ylabel(feature)
    plt.tight_layout()
    plt.show()

In [ ]:
# Correlation matrix for selected numeric features.

heatmap_cols = [
    "pT217_F", "AB42_F", "AB40_F", "NfL_Q", "GFAP_Q",
    "PHC_MEM", "PHC_EXF", "PHC_LAN",
    "Left.Hippocampus_combat", "Right.Hippocampus_combat",
    "lh_entorhinal_volume_combat", "rh_entorhinal_volume_combat",
    "Left.Lateral.Ventricle_combat", "Right.Lateral.Ventricle_combat",
    "BrainSegVol_combat", "EstimatedTotalIntraCranialVol_combat",
    "DIAGNOSIS", "Month",
]
heatmap_cols = [c for c in heatmap_cols if c in df.columns]

corr = df[heatmap_cols].corr(numeric_only=True)

plt.figure(figsize=(12, 10))
if sns is not None:
    sns.heatmap(corr, cmap="coolwarm", center=0, square=True)
else:
    plt.imshow(corr, aspect="auto")
    plt.colorbar()
    plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
    plt.yticks(range(len(corr.index)), corr.index)
plt.title("Correlation Matrix: Core Features")
plt.tight_layout()
plt.show()

## 7. Multimodal Coverage and Longitudinal Trends

This section combines the final tabular dataset with raw MRI folder counts to quantify:

- modality availability  
- MRI + tabular overlap  
- longitudinal visit coverage  
- DICOM density  
- brain atrophy trends over 36 months

In [ ]:
def count_dcm_per_patient(base_dir: Path) -> dict:
    """Count .dcm files per patient directory."""
    counts = {}
    if not base_dir.exists():
        return counts

    for patient_dir in sorted(base_dir.iterdir()):
        if patient_dir.is_dir() and not patient_dir.name.startswith("."):
            n = len(list(patient_dir.rglob("*.dcm")))
            counts[patient_dir.name] = n

    return counts

alpha_counts = count_dcm_per_patient(ALPHA_DIR)
beta_counts = count_dcm_per_patient(BETA_DIR)

print(f"ADNIAlpha: {len(alpha_counts):,} patients, {sum(alpha_counts.values()):,} total DICOMs")
print(f"ADNIBeta: {len(beta_counts):,} patients, {sum(beta_counts.values()):,} total DICOMs")

mri_counts = defaultdict(int)
for ptid, n in alpha_counts.items():
    mri_counts[ptid] += n
for ptid, n in beta_counts.items():
    mri_counts[ptid] += n

mri_ptids = set(mri_counts.keys())
tab_ptids = set(df["PTID"].unique())

both_ptids = mri_ptids & tab_ptids
tab_only_ptids = tab_ptids - mri_ptids
mri_only_ptids = mri_ptids - tab_ptids

overlap_summary = pd.DataFrame({
    "Modality Group": ["MRI + Tabular", "Tabular Only", "MRI Only"],
    "Patients": [len(both_ptids), len(tab_only_ptids), len(mri_only_ptids)],
})
display(overlap_summary)

In [ ]:
# Assign each patient a baseline diagnosis where possible.

baseline = df[df["Month"] == 0][["PTID", "DIAGNOSIS"]].drop_duplicates("PTID")
ptid_to_dx = dict(zip(baseline["PTID"], baseline["DIAGNOSIS"]))

for ptid in tab_ptids:
    if ptid not in ptid_to_dx:
        mode = df[df["PTID"] == ptid]["DIAGNOSIS"].mode()
        if len(mode) > 0:
            ptid_to_dx[ptid] = mode.iloc[0]

for ptid in mri_only_ptids:
    if ptid not in ptid_to_dx:
        ptid_to_dx[ptid] = np.nan

modality_data = []
for dx in DX_ORDER:
    dx_patients = {p for p, d in ptid_to_dx.items() if d == dx}
    mri_patients_in_class = dx_patients & both_ptids
    avg_mri = np.mean([mri_counts[p] for p in mri_patients_in_class]) if mri_patients_in_class else 0

    modality_data.append({
        "Diagnosis": DX_LABELS_LONG[dx],
        "MRI + Tabular": len(dx_patients & both_ptids),
        "Tabular Only": len(dx_patients & tab_only_ptids),
        "MRI Only": len(dx_patients & mri_only_ptids),
        "Avg MRI Images": avg_mri,
    })

df_mod = pd.DataFrame(modality_data)
display(df_mod)

In [ ]:
# Plot modality distribution by diagnosis.

plot_df = df_mod.set_index("Diagnosis")[["MRI + Tabular", "Tabular Only", "MRI Only"]]

ax = plot_df.plot(kind="bar", figsize=(9, 5))
ax.set_title("Data Modality Distribution by Diagnosis")
ax.set_xlabel("Diagnosis")
ax.set_ylabel("Patients")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Longitudinal data availability over 36 months.

month_counts = df.groupby(["Month", "Diagnosis_Label"], observed=False)["PTID"].nunique().reset_index()

plt.figure(figsize=(10, 5))
if sns is not None:
    sns.lineplot(data=month_counts, x="Month", y="PTID", hue="Diagnosis_Label", marker="o")
else:
    for label, grp in month_counts.groupby("Diagnosis_Label"):
        plt.plot(grp["Month"], grp["PTID"], marker="o", label=label)
    plt.legend()

plt.title("Longitudinal Patient Availability by Diagnosis")
plt.xlabel("Month from Baseline")
plt.ylabel("Unique Patients")
plt.tight_layout()
plt.show()

In [ ]:
# Brain atrophy trends over 36 months.

atrophy_features = [
    "Left.Hippocampus_combat",
    "Right.Hippocampus_combat",
    "lh_entorhinal_volume_combat",
    "rh_entorhinal_volume_combat",
    "Left.Lateral.Ventricle_combat",
    "Right.Lateral.Ventricle_combat",
]
atrophy_features = [c for c in atrophy_features if c in df.columns]

for feature in atrophy_features:
    trend = df.groupby(["Month", "Diagnosis_Label"], observed=False)[feature].mean().reset_index()

    plt.figure(figsize=(10, 5))
    if sns is not None:
        sns.lineplot(data=trend, x="Month", y=feature, hue="Diagnosis_Label")
    else:
        for label, grp in trend.groupby("Diagnosis_Label"):
            plt.plot(grp["Month"], grp[feature], label=label)
        plt.legend()

    plt.title(f"{feature} Trend Over 36 Months")
    plt.xlabel("Month from Baseline")
    plt.ylabel(feature)
    plt.tight_layout()
    plt.show()

In [ ]:
# MRI DICOM density per patient.

mri_density_df = pd.DataFrame({
    "PTID": list(mri_counts.keys()),
    "DICOM_Count": list(mri_counts.values()),
})

display(mri_density_df["DICOM_Count"].describe().to_frame("DICOM Count Summary"))

plt.figure(figsize=(8, 5))
plt.hist(mri_density_df["DICOM_Count"], bins=30)
plt.title("MRI DICOM Count Distribution per Patient")
plt.xlabel("DICOM files per patient")
plt.ylabel("Patients")
plt.tight_layout()
plt.show()

## 8. Final EDA Takeaways

Key findings from the merged EDA:

1. The analysis-ready dataset contains **11,631 observations**, **3,762 patients**, and a **0–36 month** longitudinal window.
2. The final compact feature set includes clinical diagnosis, APOE genotype, plasma biomarkers, cognitive scores, and selected MRI-derived regions.
3. MRI data is meaningful but sparse relative to tabular data, with **581 MRI + tabular patients** and **3,181 tabular-only patients**.
4. APOE genotype and plasma biomarkers provide strong disease-state signal.
5. Cognitive scores decline across CN → MCI → AD.
6. MRI-derived features support expected neurodegeneration patterns, including hippocampal/entorhinal changes and ventricle expansion.
7. MCI remains heterogeneous, which should be considered in downstream modeling.